In [3]:
import pandas as pd
import kagglehub
import os
from pathlib import Path

# 1. Definindo os caminhos das pastas (Arquitetura de Medalhão)
silver_path = Path("Data/Silver")
silver_path.mkdir(parents=True, exist_ok=True)

print("🛰️ Baixando dataset de jogadores do Kaggle...")
# 2. Download do dataset (Sempre pega a versão mais recente)
path_raw = kagglehub.dataset_download("hubertsidorowicz/football-players-stats-2025-2026")
csv_file = os.path.join(path_raw, "players_data_light-2025_2026.csv")

# 3. Lendo e Filtrando
print("🇮🇹 Filtrando jogadores da 'it Serie A'...")
df_raw = pd.read_csv(csv_file)
df_serie_a = df_raw[df_raw['Comp'] == 'it Serie A'].copy()

# 4. Criando a coluna de 'Date' (Evitando o erro do sort_values)
# Como o CSV original é um resumo, usamos o Rank (Rk) para simular o tempo
if 'Date' not in df_serie_a.columns:
    df_serie_a['Date'] = pd.to_datetime('2026-01-01') + pd.to_timedelta(df_serie_a['Rk'], unit='D')

# 5. Salvando na Camada Silver (Parquet é vida!)
output_file = silver_path / "serie_a_silver.parquet"
df_serie_a.to_parquet(output_file, index=False)

print(f"✨ Sucesso! O df_serie_a foi criado com {len(df_serie_a)} jogadores.")
print(f"📁 Salvo em: {output_file}")

# Exibindo os primeiros registros para conferir
df_serie_a.head()

c:\Users\kaiki\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🛰️ Baixando dataset de jogadores do Kaggle...
🇮🇹 Filtrando jogadores da 'it Serie A'...
✨ Sucesso! O df_serie_a foi criado com 578 jogadores.
📁 Salvo em: Data\Silver\serie_a_silver.parquet


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


#2 Dados StatsBomb

In [5]:
# Carregar dados de eventos do StatsBomb (CSV ou JSON)
import glob
import json

path_sb = kagglehub.dataset_download("saurabhshahane/statsbomb-football-data")
file_sb = os.path.join(path_sb, "events.csv")

if os.path.exists(file_sb):
    df_sb = pd.read_csv(file_sb)
    print("\n" + "="*60)
    print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - CSV)")
    print("="*60)
    print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
    print("\n--- AMOSTRA DOS DADOS ---")
    print(df_sb.head())
else:
    # Fallback para o formato real desta base: JSONs em data/events
    event_json_files = glob.glob(os.path.join(path_sb, "**", "events", "*.json"), recursive=True)
    print(f"events.csv não encontrado. JSONs de eventos encontrados: {len(event_json_files)}")

    if not event_json_files:
        print(f"Nenhum arquivo de eventos foi encontrado em {path_sb}")
    else:
        records = []
        max_files = 20  # amostra para carregar rápido
        for fp in event_json_files[:max_files]:
            with open(fp, "r", encoding="utf-8") as f:
                events = json.load(f)
            if isinstance(events, list):
                records.extend(events)

        if records:
            df_sb = pd.json_normalize(records)
            print("\n" + "="*60)
            print("ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)")
            print("="*60)
            print(f"Linhas: {df_sb.shape[0]:,} | Colunas: {df_sb.shape[1]}")
            print("\nPrimeiras colunas:")
            print(df_sb.columns[:20].tolist())
            print("\n--- AMOSTRA DOS DADOS ---")
            print(df_sb.head())
        else:
            print("JSONs encontrados, mas nenhum registro válido foi carregado.")

df_serie_a.head()

events.csv não encontrado. JSONs de eventos encontrados: 3464

ESTRUTURA DO DATASET STATSBOMB (EVENTOS - JSON)
Linhas: 75,951 | Colunas: 137

Primeiras colunas:
['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type.id', 'type.name', 'possession_team.id', 'possession_team.name', 'play_pattern.id', 'play_pattern.name', 'team.id', 'team.name', 'tactics.formation', 'tactics.lineup', 'related_events', 'location']

--- AMOSTRA DOS DADOS ---
                                     id  index  period     timestamp  minute  \
0  9f6e2ecf-6685-45df-a62e-c2db3090f6c1      1       1  00:00:00.000       0   
1  0300039d-150d-41e4-b29a-76602ef002e6      2       1  00:00:00.000       0   
2  491e8901-7630-4cc8-b57b-937dddff2eaa      3       1  00:00:00.000       0   
3  757b85ad-ddfe-44d5-b893-c23a9fb709d8      4       1  00:00:00.000       0   
4  549567bd-36de-4ac8-b8dc-6b5d3f1e4be8      5       1  00:00:00.575       0   

   second  possession  duration  type.id   

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Date
12,13,Zakaria Aboukhlal,ma MAR,MF,Torino,it Serie A,26.0,2000.0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-14
18,19,Francesco Acerbi,it ITA,DF,Inter,it Serie A,38.0,1988.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-20
22,23,Che Adams,sct SCO,FW,Torino,it Serie A,29.0,1996.0,26,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-24
26,27,Jayden Addai,nl NED,MF,Como,it Serie A,20.0,2005.0,12,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28
33,34,Vasilije Adžić,me MNE,MF,Juventus,it Serie A,19.0,2006.0,10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-04


In [ ]:
# Diagnóstico: localizar automaticamente arquivos de evento no dataset StatsBomb
import glob

csv_files = glob.glob(os.path.join(path_sb, "**", "*.csv"), recursive=True)
json_files = glob.glob(os.path.join(path_sb, "**", "*.json"), recursive=True)

print("CSV encontrados:", len(csv_files))
print("JSON encontrados:", len(json_files))

# Mostrar alguns caminhos para inspeção
print("\nExemplos de CSV:")
for p in csv_files[:20]:
    print("-", p)

# Tentar achar automaticamente um arquivo de eventos
event_candidates = [p for p in csv_files if "event" in os.path.basename(p).lower() or "events" in p.lower()]

if event_candidates:
    chosen_file = event_candidates[0]
    print(f"\nArquivo de eventos escolhido automaticamente: {chosen_file}")
    df_sb = pd.read_csv(chosen_file)
    print(f"df_sb carregado com sucesso -> Linhas: {df_sb.shape[0]:,}, Colunas: {df_sb.shape[1]}")
else:
    print("\nNenhum CSV de eventos encontrado automaticamente.")
    print("Pode ser que este dataset esteja em JSON; nesse caso podemos montar um parser na próxima célula.")

In [10]:
import os
import json
import pandas as pd

# 1. Lista das ligas que você quer comparar (IDs do StatsBomb)
# Bundesliga (9), La Liga (11), World Cup (43), Champions League (16)
competicoes_interesse = [9, 11, 43, 16]

resumo_dados = []

for _, row in df_comps.iterrows():
    if row['competition_id'] in competicoes_interesse:
        comp_id = row['competition_id']
        season_id = row['season_id']
        
        # Caminho para a pasta de matches desta competição/temporada
        path_matches = os.path.join(path_sb, "data", "matches", str(comp_id), f"{season_id}.json")
        
        if os.path.exists(path_matches):
            with open(path_matches, "r", encoding="utf-8") as f:
                matches = json.load(f)
                num_jogos = len(matches)
                
                resumo_dados.append({
                    'Liga': row['competition_name'],
                    'Temporada': row['season_name'],
                    'Qtd_Jogos': num_jogos,
                    'Comp_ID': comp_id,
                    'Season_ID': season_id
                })

# 2. Criar um ranking de volume
df_ranking = pd.DataFrame(resumo_dados).sort_values(by='Qtd_Jogos', ascending=False)

print("📊 VOLUME DE DADOS POR LIGA (StatsBomb):")
print(df_ranking.to_string(index=False))

📊 VOLUME DE DADOS POR LIGA (StatsBomb):
            Liga Temporada  Qtd_Jogos  Comp_ID  Season_ID
         La Liga 2015/2016        380       11         27
   1. Bundesliga 2015/2016        306        9         27
  FIFA World Cup      2022         64       43        106
  FIFA World Cup      2018         64       43          3
         La Liga 2014/2015         38       11         26
         La Liga 2011/2012         37       11         23
         La Liga 2017/2018         36       11          1
         La Liga 2020/2021         35       11         90
         La Liga 2009/2010         35       11         21
   1. Bundesliga 2023/2024         34        9        281
         La Liga 2018/2019         34       11          4
         La Liga 2016/2017         34       11          2
         La Liga 2019/2020         33       11         42
         La Liga 2010/2011         33       11         22
         La Liga 2012/2013         32       11         24
         La Liga 2013/2014      

In [ ]:
# Carregar eventos a partir dos JSONs do StatsBomb
import json

# Arquivos JSON de eventos (geralmente ficam em pasta 'events')
event_json_files = [p for p in json_files if "events" in p.lower() and p.lower().endswith(".json")]

print(f"Arquivos JSON de eventos encontrados: {len(event_json_files)}")
for p in event_json_files[:5]:
    print("-", p)

# Limitar quantidade para análise inicial (rápido)
MAX_FILES = 50
selected_files = event_json_files[:MAX_FILES]

records = []
for fp in selected_files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            events = json.load(f)
        if isinstance(events, list):
            records.extend(events)
    except Exception as e:
        print(f"Falha ao ler {fp}: {e}")

if records:
    df_sb = pd.json_normalize(records)
    print(f"\ndf_sb carregado de JSON com sucesso -> Linhas: {df_sb.shape[0]:,}, Colunas: {df_sb.shape[1]}")
    print("\nColunas principais:")
    print(df_sb.columns[:20].tolist())
else:
    print("Nenhum evento foi carregado dos arquivos JSON selecionados.")

##teste

In [6]:
import json
import pandas as pd
import os
from pathlib import Path

# 1. Localizar o arquivo de Competitions (O Catálogo)
path_competitions = os.path.join(path_sb, "data", "competitions.json")

with open(path_competitions, "r", encoding="utf-8") as f:
    comps = json.load(f)

df_comps = pd.json_normalize(comps)

# 2. Filtrar pela temporada mais recente (ex: 2020/2021 ou a maior que tiver)
# Ordenamos por season_id para pegar o que há de mais novo no dataset
df_recent_comps = df_comps.sort_values(by="season_id", ascending=False)

print("🏆 Temporadas mais recentes encontradas:")
print(df_recent_comps[['competition_name', 'season_name', 'competition_id', 'season_id']].head(5))

# 3. Pegar os IDs da temporada mais recente disponível (ex: La Liga 2020/2021)
target_comp = df_recent_comps.iloc[0]['competition_id']
target_season = df_recent_comps.iloc[0]['season_id']

# 4. Localizar os Matches dessa temporada específica
path_matches = os.path.join(path_sb, "data", "matches", str(target_comp), f"{target_season}.json")

with open(path_matches, "r", encoding="utf-8") as f:
    matches = json.load(f)

# Pegamos os IDs de todos os jogos dessa temporada "Elite"
match_ids = [m['match_id'] for m in matches]
print(f"✅ Encontrados {len(match_ids)} jogos recentes da temporada {df_recent_comps.iloc[0]['season_name']}.")

🏆 Temporadas mais recentes encontradas:
     competition_name season_name  competition_id  season_id
71  UEFA Women's Euro        2025              53        315
21       Copa America        2024             223        282
68          UEFA Euro        2024              55        282
0       1. Bundesliga   2023/2024               9        281
24       Copa del Rey   1977/1978              87        279
✅ Encontrados 31 jogos recentes da temporada 2025.


In [8]:
records = []
MAX_JOGOS = 3 # Ajuste conforme sua RAM permitir

for mid in match_ids[:MAX_JOGOS]:
    file_path = os.path.join(path_sb, "data", "events", f"{mid}.json")
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            events = json.load(f)
            # Injetamos o match_id para controle
            for e in events: e['match_id'] = mid
            records.extend(events)

df_sb_recent = pd.json_normalize(records)
print(f"💎 Silver Layer atualizada com {df_sb_recent.shape[0]:,} eventos da temporada mais recente!")

💎 Silver Layer atualizada com 13,313 eventos da temporada mais recente!
